# Private Credit Risk Measures — Student Version

Complete the marked tasks for a simplified Pye-Barker Fire & Safety loan through **cash flows → price sensitivity → DTS**.

### Given assumptions

- **Publicly disclosed terms:** Pye-Barker Fire & Safety, LLC first-lien debt at SOFR + 450 bps, a 75 bp reference-rate floor, and maturity on May 26, 2031.
- **Synthetic teaching scenario:** 5.5 years remaining, quarterly payments and principal amortisation of 1% per annum, paid quarterly, based on original principal.
- **Synthetic curves:** SOFR forwards at 4.25%; non-flat all-in discount zero rates around 8.40%–8.83%, with quarterly compounding.
- No call, PIK, cash sweep or default is modelled; the floor is inactive and omitted.

Amounts are per **100 of original principal**. The borrower terms above are from a public [SEC Form 10-Q filing](https://www.sec.gov/Archives/edgar/data/1973476/000197347624000074/pif-20240630.htm). All remaining model inputs and scenarios are synthetic classroom assumptions, not market observations or investment advice.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

MARGIN_BPS = 450
TENOR_YRS = 5.5
FREQ = 4
AMORT_PCT = 0.01
FIXED_CURRENT_RATE = 0.0425
BUMP_BPS = 1.0

print(f"Coupon: SOFR + {MARGIN_BPS} bps | Horizon: {TENOR_YRS:.1f} years")
print(f"Payments/year: {FREQ} | Annual amortisation: {AMORT_PCT:.0%}")

## 1 — Given curves, cash flows and price

SOFR forwards project future coupons. The supplied all-in loan discount zero rates discount those payments directly.

$$P=\sum_i\frac{CF_i}{(1+y_i/f)^{ft_i}}.$$
The coupon margin remains fixed at 450 bps, and the current coupon uses its already-fixed SOFR rate.

### Task 1
Complete the quarterly coupon calculation in `build_cashflows()`. The principal amortisation and final balloon payment are supplied.

In [ ]:
# GIVEN synthetic curves; no bootstrap or spread calibration.
times = np.arange(1, int(round(TENOR_YRS * FREQ)) + 1) / FREQ
sofr_curve = np.full(times.shape, 0.0425)
node_years = np.array([0.25, 0.50, 1.00, 2.00, 3.00, 4.00, 5.50])
node_discount_pct = np.array([8.40, 8.52, 8.71, 8.83, 8.79, 8.71, 8.750515])
discount_curve = np.interp(times, node_years, node_discount_pct / 100)

def discount_factors(zero_curve, times, freq):
    return (1 + np.asarray(zero_curve) / freq) ** (-np.asarray(times) * freq)

given_curves = pd.DataFrame({
    "payment (years)": times,
    "SOFR forward (%)": sofr_curve * 100,
    "loan discount zero (%)": discount_curve * 100,
    "loan DF": discount_factors(discount_curve, times, FREQ),
})
node_rows = [int(round(year * FREQ)) - 1 for year in node_years]
print(given_curves.iloc[node_rows].to_string(
    index=False, formatters={"loan DF": "{:.4f}".format}))

In [ ]:
def build_cashflows(sofr_forwards, margin_bps, amort_pct, freq, fixed_current_rate):
    rates = np.array(sofr_forwards, dtype=float, copy=True)
    rates[0] = fixed_current_rate
    cfs = np.zeros(len(rates))
    outstanding = 100.0
    amort_per_period = 100.0 * amort_pct / freq

    for index, rate in enumerate(rates):
        # TODO: calculate the quarterly coupon from outstanding principal,
        # the reference rate, contractual margin and payment frequency.
        raise NotImplementedError("Complete the coupon calculation")

        if index < len(rates) - 1:
            cfs[index] = coupon + amort_per_period
            outstanding -= amort_per_period
        else:
            cfs[index] = coupon + outstanding
    return cfs

def price(cfs, times, discount_curve, freq):
    return float(np.sum(cfs * discount_factors(discount_curve, times, freq)))

cf = build_cashflows(sofr_curve, MARGIN_BPS, AMORT_PCT, FREQ, FIXED_CURRENT_RATE)
p0 = price(cf, times, discount_curve, FREQ)
print(f"Periods: {len(cf)} | Base price: {p0:.4f}")
cashflow_view = pd.DataFrame({"payment (years)": times, "cash flow": cf})
print(cashflow_view.iloc[[0, 1, 2, -1]].to_string(index=False))

## 2 — Calculate rate duration and spread duration

$$D\approx-\frac{P(+h)-P(-h)}{2hP_0},\qquad h=0.0001\text{ for 1 bp}.$$
| Calculation | Bump by ±1 bp | Hold fixed |
|---|---|---|
| **Rate duration — common rate shift** | Both SOFR forwards and discount zero rates; recalculate future coupons | Current fixing and contractual margin |
| **Spread duration** | Discount zero rates only | SOFR forwards, contractual margin and all cash flows |

### Task 2
Complete both functions. Pay particular attention to whether cash flows should be rebuilt after each bump.

In [ ]:
def rate_duration(sofr_curve, discount_curve, times, margin_bps, amort_pct,
                  freq, fixed_current_rate, bump_bps=BUMP_BPS):
    h = bump_bps / 1e4
    # TODO: rebuild cash flows after shifting both the SOFR forward curve
    # and the discount curve up and down by h, then apply central difference.
    raise NotImplementedError("Complete rate_duration")

def spread_duration(cfs, times, discount_curve, freq, bump_bps=BUMP_BPS):
    h = bump_bps / 1e4
    # TODO: keep cfs fixed, shift only the discount curve up and down by h,
    # then apply central difference.
    raise NotImplementedError("Complete spread_duration")

d_rate = rate_duration(sofr_curve, discount_curve, times, MARGIN_BPS,
                       AMORT_PCT, FREQ, FIXED_CURRENT_RATE)
d_sprd = spread_duration(cf, times, discount_curve, FREQ)
print(f"Rate duration (common shift): {d_rate:.2f}")
print(f"Spread duration:             {d_sprd:.2f}")

### Interpret your results

1. Which duration is larger?
2. Why do changing future coupons offset part of the rate-duration price effect?
3. Why must the same cash flows be used for all three spread-duration prices?

## 3 — Convert spread duration to DTS

$$\frac{\Delta P}{P}\approx-D^{spread}\Delta s=-(D^{spread}s)\frac{\Delta s}{s}.$$
Assume a valuation spread of **450 bps** for this illustration. This is an additional DTS input, not the coupon margin and not a spread calibrated from the supplied discount curve.

### Task 3
Complete `dts()` and estimate the first-order price loss for a 10% relative spread widening. Use spread in percentage points: 450 bps becomes 4.5.

In [ ]:
def dts(spread_duration_yrs, spread_bps):
    # TODO: convert spread_bps to percentage points and multiply by duration.
    raise NotImplementedError("Complete dts")

DTS_SPREAD_BPS = 450
relative_widening = 0.10
loan_dts = dts(d_sprd, DTS_SPREAD_BPS)
estimated_loss_pct = loan_dts * relative_widening

print(f"DTS: {loan_dts:.2f}")
print(f"10% relative spread widening | First-order price loss: {estimated_loss_pct:.2f}%")

## Submission checks

Before submitting, confirm that:

- The first coupon retains the current fixing when the SOFR curve is bumped.
- Principal repayments sum to 100, including the final balloon payment.
- Rate duration rebuilds future floating-rate coupons.
- Spread duration passes the same cash-flow array to its base, up and down prices.
- The assumed 450 bp valuation spread affects DTS only.